In [23]:
import numpy as np
from numba import njit
from itertools import product

OPEN = (6, 7, 8)

def actions_from_roll(d1, d2, d3, d4):
    """ Returns up to 3 unique actions (dx,dy,dz) from the 3 pairings. """
    pairings = ((d1+d2, d3+d4), (d1+d3, d2+d4), (d1+d4, d2+d3))
    acts = set()
    for s1, s2 in pairings:
        dx = (1 if s1 == 6 else 0) + (1 if s2 == 6 else 0)
        dy = (1 if s1 == 7 else 0) + (1 if s2 == 7 else 0)
        dz = (1 if s1 == 8 else 0) + (1 if s2 == 8 else 0)
        if dx + dy + dz > 0:
            acts.add((dx, dy, dz))
    return list(acts)

def precompute_actions():
    """ Builds a padded action array of shape (1296, 3, 3) with -1 (because numba requires fixed size arrays). """
    rolls = list(product(range(1, 7), repeat=4))  # 6^4 = 1296 possible rolls with 4 dice
    R = len(rolls)
    Amax = 3
    actions = -np.ones((R, Amax, 3), dtype=np.int8) # size = 1296*3*3

    for r, (d1, d2, d3, d4) in enumerate(rolls):
        acts = actions_from_roll(d1, d2, d3, d4)
        for k in range(min(len(acts), Amax)):
            actions[r, k, 0] = acts[k][0]
            actions[r, k, 1] = acts[k][1]
            actions[r, k, 2] = acts[k][2]

    probs = np.full(R, 1.0 / R)  # uniform over all (ordered) rolls
    return actions, probs

@njit
def compute_V(N6, N7, N8, actions, probs):
    
    V = np.zeros((N6 + 1, N7 + 1, N8 + 1))
    V[N6, N7, N8] = N6 + N7 + N8

    Smax = N6 + N7 + N8
    R = actions.shape[0]
    Amax = actions.shape[1]

    for s in range(Smax - 1, -1, -1):
        # iterates all (x,y,z) with x+y+z = s within bounds
        
        # N6 
        x_min = max(0, s - (N7 + N8)) # y+z = s-x 
        x_max = min(N6, s)
        for x in range(x_min, x_max + 1):
            dif = s - x
            y_min = max(0, dif - N8)
            y_max = min(N7, dif)
            
            # N7
            for y in range(y_min, y_max + 1):
                z = dif - y
                if z < 0 or z > N8:
                    continue
                if x == N6 and y == N7 and z == N8:
                    continue

                immediate_gain = x + y + z  # if we stop

                # if we keep going
                future_gain = 0.0
                for r in range(R):
                    best = 0.0  # if no admissible action, we get 0
                    for k in range(Amax):
                        dx = actions[r, k, 0]
                        if dx < 0:
                            continue
                        dy = actions[r, k, 1]
                        dz = actions[r, k, 2]
                        nx = x + dx
                        ny = y + dy
                        nz = z + dz
                        if nx <= N6 and ny <= N7 and nz <= N8: # checks admissibility
                            val = V[nx, ny, nz]
                            if val > best:
                                best = val
                    future_gain += probs[r] * best

                V[x, y, z] = immediate_gain if immediate_gain >= future_gain else future_gain

    return V

In [24]:
N6, N7, N8 = 100, 100, 100
actions, probs = precompute_actions()
V = compute_V(N6, N7, N8, actions, probs)
print("Le gain optimal est :", V[0, 0, 0])

Le gain optimal est : 6.330720105660973


In [25]:
N6, N7, N8 = 10, 10, 10
actions, probs = precompute_actions()
V = compute_V(N6, N7, N8, actions, probs)
print("Le gain optimal est :", V[0, 0, 0])

Le gain optimal est : 6.328950820370477


In [26]:
N6, N7, N8 = 10, 10, 100
actions, probs = precompute_actions()
V = compute_V(N6, N7, N8, actions, probs)
print("Le gain optimal est :", V[0, 0, 0])

Le gain optimal est : 6.329833382425647


In [27]:
N6, N7, N8 = 0, 10, 10
actions, probs = precompute_actions()
V = compute_V(N6, N7, N8, actions, probs)
print("Le gain optimal est :", V[0, 0, 0])

Le gain optimal est : 1.6737422485811653


In [28]:
N6, N7, N8 = 10, 0, 10
actions, probs = precompute_actions()
V = compute_V(N6, N7, N8, actions, probs)
print("Le gain optimal est :", V[0, 0, 0])

Le gain optimal est : 1.5628329146210898
